In [8]:
import pandas as pd
import numpy as np


In [9]:
# Read files saved

text_assigned = pd.read_csv("data/text_reassigned_memes.csv", low_memory=False)
new_meme_data = pd.read_csv("data/new_assigned_memes.csv", low_memory=False)
# img_reassigned = pd.read_csv("data/reassigned_tpl_with_images.csv")

new_meme_data


,key,text_for_embed,final_from_text,text_sim_top1,text_margin,global_context_description,local_context_user_texts,local_context_text_meaning,local_context_instance_specific_image_description,global_context_keywords,...,local_context_meme_template_overlay,global_context_thought,confidence,phash_dist,clip_top1,clip_margin,match_method,template,text_template,img_template
0,meme_submissions_1343519,"A cat with a loading symbol on its forehead, l...",Loading Cat,0.924177,0.026697,"A cat with a loading symbol on its forehead, l...",['Hitler when he saw a blue-eyed Jew'],The meme humorously depicts Hitler's supposed ...,NaN,"['cat', 'loading symbol', 'confusion', 'distre...",...,NaN,NaN,0.791371,12.0,0.791371,0.002385,none,NO_TEMPLATE,Loading Cat,NO_TEMPLATE
1,meme_submissions_134352,A three-panel meme format. The first panel sho...,NaN,NaN,NaN,A three-panel meme format. The first panel sho...,[],The meme humorously depicts a character who cl...,The second panel contains an image of several ...,"['fear', 'unscared', 'scared', 'meme format', ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,I fear no man. But that thing... it scares me.,I fear no man. But that thing... it scares me.,I fear no man. But that thing... it scares me.
2,meme_submissions_1343524,A comparison meme showing two fictional creatu...,NO_TEMPLATE,0.653213,0.076674,A comparison meme showing two fictional creatu...,[],The meme highlights the similarities between t...,NaN,"['comparison', 'creatures', 'minecraft', 'stra...",...,NaN,NaN,0.722954,12.0,0.722954,0.018458,none,NO_TEMPLATE,NO_TEMPLATE,NO_TEMPLATE
3,meme_submissions_1343526,A four-panel meme format featuring Homer Simps...,NaN,NaN,NaN,A four-panel meme format featuring Homer Simps...,"['Increase carbon filtering', 'produce more wi...",The meme criticizes the perceived ineffectiven...,NaN,"['Homer Simpson', 'The Simpsons', 'stupid acti...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Homer Simpson ""Something so stupid""","Homer Simpson ""Something so stupid""","Homer Simpson ""Something so stupid"""
4,meme_submissions_134353,The meme shows a comparison between a house co...,NO_TEMPLATE,0.615970,0.009624,The meme shows a comparison between a house co...,['My house coat in the morning vs my house coa...,The meme humorously exaggerates the difference...,The image is split into two parts. The top tex...,"['house coat', 'morning', 'night', 'comparison...",...,NaN,NaN,0.755211,14.0,0.755211,0.002414,none,NO_TEMPLATE,NO_TEMPLATE,NO_TEMPLATE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171788,meme_submissions_114709,A three-panel image. The left column contains ...,NO_TEMPLATE,0.606668,0.001281,A three-panel image. The left column contains ...,[],The meme humorously contrasts the perceived gr...,NaN,"['three panel', 'throne', 'chief', 'king', 'go...",...,NaN,NaN,0.804002,14.0,0.804002,0.008509,none,NO_TEMPLATE,NO_TEMPLATE,NO_TEMPLATE
171789,meme_submissions_1147093,A four-panel meme format. The top left panel c...,NO_TEMPLATE,0.692856,0.024583,A four-panel meme format. The top left panel c...,"['FLY TRYING TO GET OUT', 'FLY ENTERING MY HOU...",The meme humorously contrasts the difficulty a...,NaN,"['Bart Simpson', 'meme format', 'blindfolded',...",...,NaN,NaN,0.699996,14.0,0.699996,0.000406,none,NO_TEMPLATE,NO_TEMPLATE,NO_TEMPLATE
171790,meme_submissions_1147095,A meme format featuring a two-panel image. The...,NaN,NaN,NaN,A meme format featuring a two-panel image. The...,['FORTNITE'],The meme expresses disappointment that the cha...,The top panel shows the animated character Ric...,"['disappointment', 'corruption', 'degradation'...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Look how they massacred my boy,Look how they massacred my boy,Look how they massacred my boy
171791,meme_submissions_1147096,A two-panel meme format featuring the characte...,NaN,NaN,NaN,A two-panel meme format featuring the characte...,"['THE FOOD\nIN THE ADS', 'WHAT THEY\nATUALLY L...",This meme contrasts the idealized presentation...,N

In [4]:
# KEY = "key"

# def preprocess_short(new_meme_data: pd.DataFrame,
#                      text_df: pd.DataFrame,
#                      img_df: pd.DataFrame) -> pd.DataFrame:
#     # ensure key is string
#     for df in (new_meme_data, text_df, img_df):
#         df[KEY] = df[KEY].astype(str)

#     # keep only needed columns and dedupe by key
#     text_keep = (
#         text_df[[KEY, "text_sim_top1", "text_margin"]]
#         .drop_duplicates(KEY)
#         .rename(columns={
#             "text_sim_top1": "text_sim_top1",
#             "text_margin": "text_margin"
#         })
#     )

#     img_keep = (
#         img_df[[KEY, "match_method", "confidence", "phash_dist", "clip_top1", "clip_margin"]]
#         .drop_duplicates(KEY)
#         .rename(columns={
#             "confidence": "img_confidence"
#         })
#     )

#     # merge into the newest dataset
#     out = (new_meme_data
#            .merge(text_keep, on=KEY, how="left")
#            .merge(img_keep, on=KEY, how="left"))

#     return out


In [5]:
# new_meme_data = preprocess_short(new_meme_data, text_assigned, img_reassigned)
# new_meme_data.head()

In [10]:
NO = "NO_TEMPLATE"
base_no = new_meme_data["template"].eq(NO)

# Only count rows as "new candidate template" if the suggested template is present and not NO_TEMPLATE
new_by_text = (
    base_no
    & new_meme_data["text_template"].notna()
    & new_meme_data["text_template"].ne(NO)
)

new_by_img = (
    base_no
    & new_meme_data["img_template"].notna()
    & new_meme_data["img_template"].ne(NO)
)

text_only_new = new_meme_data[new_by_text & ~new_by_img].copy()
img_only_new = new_meme_data[~new_by_text & new_by_img].copy()
both_new = new_meme_data[new_by_text & new_by_img].copy()

both_new["agree"] = both_new["text_template"] == both_new["img_template"]

print("text_only_new:", len(text_only_new))
print("img_only_new:", len(img_only_new))
print("both_new:", len(both_new))
print("both_new agree:", both_new["agree"].sum(), "| disagree:", (~both_new["agree"]).sum())


text_only_new: 6142
img_only_new: 4832
both_new: 3232
both_new agree: 1216 | disagree: 2016


In [11]:
# Build confidence bins for TEXT, based on top 1 sim score, and margin
def text_conf_bin(df):
    sim = df["text_sim_top1"]
    mar = df["text_margin"]

    return np.select(
        [
            (sim >= 0.70) & (mar >= 0.06),  # very safe
            (sim >= 0.68) & (mar >= 0.04),  # safe
            (sim >= 0.64) & (mar >= 0.04),  # looser
        ],
        ["high", "mid", "low_plus"],
        default="low"
    )

for d in (text_only_new, both_new):
    d["text_conf_bin"] = text_conf_bin(d)

text_only_new["text_conf_bin"].value_counts(dropna=False)


text_conf_bin
low     3061
high    2173
mid      908
Name: count, dtype: int64

In [13]:
# Build confidence bins for IMAGE, based on pHash Distance + CLIP Embeddings
def img_conf_bin(df):
    ph = df["phash_dist"] if "phash_dist" in df.columns else pd.Series(np.nan, index=df.index)
    ct = df["clip_top1"] if "clip_top1" in df.columns else pd.Series(np.nan, index=df.index)
    cm = df["clip_margin"] if "clip_margin" in df.columns else pd.Series(np.nan, index=df.index)

    return np.select(
        [
            ph.notna() & (ph <= 6),                         # high precision duplicate-ish
            ct.notna() & (ct >= 0.86) & (cm >= 0.03),       # strong CLIP match
            ct.notna() & (ct >= 0.82) & (cm >= 0.02),       # moderate CLIP match
        ],
        ["high_phash", "high_clip", "mid_clip"],
        default="low"
    )

for d in (img_only_new, both_new):
    d["img_conf_bin"] = img_conf_bin(d)

img_only_new["img_conf_bin"].value_counts(dropna=False)

img_conf_bin
high_phash    4154
high_clip      678
Name: count, dtype: int64

In [14]:
def strat_sample(df, n, group_col, seed=42):
    if len(df) == 0:
        return df.copy()

    rng = np.random.default_rng(seed)
    counts = df[group_col].value_counts(dropna=False)
    total = len(df)

    # proportional target per group (at least 1 if group exists)
    targets = (counts / total * n).round().astype(int)
    targets = targets.clip(lower=1)

    # don't request more than group size
    targets = targets.combine(counts, min)

    pieces = []
    for g, k in targets.items():
        gdf = df[df[group_col] == g]
        # sample without replacement
        idx = rng.choice(gdf.index.to_numpy(), size=int(k), replace=False)
        pieces.append(df.loc[idx])

    out = pd.concat(pieces, axis=0)

    # if we overshot/undershot due to rounding, fix size
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)
    elif len(out) < n:
        remaining = df.drop(index=out.index, errors="ignore")
        if len(remaining) > 0:
            add_n = min(n - len(out), len(remaining))
            out = pd.concat([out, remaining.sample(n=add_n, random_state=seed)], axis=0)

    return out


In [15]:
sample_text_only = strat_sample(text_only_new, n=150, group_col="text_conf_bin")
sample_img_only = strat_sample(img_only_new, n=60, group_col="img_conf_bin")

both_agree = both_new[both_new["agree"]].copy()
both_disagree = both_new[~both_new["agree"]].copy()

review_df = pd.concat(
    [text_only_new, img_only_new, both_agree, both_disagree],
    ignore_index=True
).drop_duplicates(subset=["key"])

print("Total to review:", len(review_df))


Total to review: 14206


In [16]:
cols = [
    "key",
    "template",
    "text_template",
    "img_template",
    "text_sim_top1",
    "text_margin",
    "text_conf_bin",
    "match_method",
    "img_confidence",
    "phash_dist",
    "clip_top1",
    "clip_margin",
    "img_conf_bin",
]

label_sheet = review_df[[c for c in cols if c in review_df.columns]].copy()

# Match the current Streamlit app schema
label_sheet["is_correct_text"] = pd.NA
label_sheet["is_correct_img"] = pd.NA

label_sheet.to_csv("annotations/template_label_sheet.csv", index=False)
print("Saved: annotations/template_label_sheet.csv")
label_sheet.head()


Saved: annotations/template_label_sheet.csv


,key,template,text_template,img_template,text_sim_top1,text_margin,text_conf_bin,match_method,phash_dist,clip_top1,clip_margin,img_conf_bin,is_correct_text,is_correct_img
0,meme_submissions_1343519,NO_TEMPLATE,Loading Cat,NO_TEMPLATE,0.924177,0.026697,low,none,12.0,0.791371,0.002385,NaN,<NA>,<NA>
1,meme_submissions_1343633,NO_TEMPLATE,"Willem Dafoe ""You can't do this to me""",NO_TEMPLATE,0.846377,0.026964,low,none,8.0,0.666739,0.022094,NaN,<NA>,<NA>
2,meme_submissions_1343705,NO_TEMPLATE,Lion climbing arrow,NO_TEMPLATE,0.832605,0.319191,high,none,10.0,0.775117,0.013009,NaN,<NA>,<NA>
3,meme_submissions_1343741,NO_TEMPLATE,Walter White Holding Gun to Head,NO_TEMPLATE,0.816156,0.026167,low,none,16.0,0.703439,0.004007,NaN,<NA>,<NA>
4,meme_submissions_1343786,NO_TEMPLATE,Sad Russian Tank Noises,NO_TEMPLATE,0.831762,0.011245,low,none,10.0,0.751891,0.028498,NaN,<NA>,<NA>
